# **HEADLINE** [change filename to descriptive name]

# Loading dataset

In [ ]:
# ==== IMPORSTS & SETTINGS ====
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# ==== FUNCTION ====
def load_csv_to_df(csv_path, sep=";"):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8", sep=sep)
        print(f"Successfully loaded CSV from {csv_path}")
        print(f"DataFrame shape: {df.shape}")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading CSV from {csv_path}: {e}")
        return None

def load_parquet_to_df(parquet_path, na=False):
    try:
        df = pd.read_parquet(parquet_path)
        print(f"Successfully loaded Parquet from {parquet_path}")
        print(f"DataFrame shape: {df.shape}")
        if na:
            print(f"DataFrame N/A counts:\n{df.isna().sum()}\n")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading Parquet from {parquet_path}: {e}")
        return None

# ==== DEFINE PATHS ====
data_path = "../Data/gcp_order/dtu_findit/extraction_and_processing/"

# ==== DEFINE FILES ====
file1 = "thesis_meta_added_equations_to_olivers_26032026.parquet"
file2 = "thesis_meta_added_linguistics_to_olivers_26032026.parquet"
file3 = "thesis_meta_added_supervisors_26032026.parquet"
file4 = "thesis_meta_all_metrics_except_grade_27032026.parquet"
file5 = "thesis_meta_all_metrics_except_grade_filtered_27032026.parquet"

# ==== LOAD DATAFRAMES ====
#print("df1:")
#df1 = load_parquet_to_df(data_path + file1)
#print("df2:")
#df2 = load_parquet_to_df(data_path + file2)
#print("df3:")
#df3 = load_parquet_to_df(data_path + file3)
df_all = load_parquet_to_df(data_path + file4)
df_filtered = load_parquet_to_df(data_path + file5)

# ==== DROP NA COLUMNS ====
df_all_noNA = df_all.dropna(axis=1, how="all")
print(f"df_all_noNA shape: {df_all_noNA.shape}")
print(f"df_all_noNA columns: {df_all_noNA.columns.tolist()}\n")
df_filtered_noNA = df_filtered.dropna(axis=1, how="all")
print(f"df_filtered_noNA shape: {df_filtered_noNA.shape}")
print(f"df_filtered_noNA columns: {df_filtered_noNA.columns.tolist()}\n")

# ==== COLUMNS TO DROP ====
drop_columns = [
    "access_ss",
    "Affiliations",
    "collection_facet",
    "format",
    "fulltext_availability_facet",
    "ISBN",
    "Journal Page",
    "isolanguage_facet",
    "Publisher",
    "Source",
    "source_all_ss",
    "match_trigger",
    "equation_pipeline_version",
    "pdf_file_analysis",
    "num_tot_pages_analysis",
    "num_cont_pages_analysis",
    "num_words_full_analysis",
    "num_words_cont_analysis",
    "abstract_ts_analysis",
    "Author_analysis",
    "Publication Year_analysis",
    "primary_member_id_s_analysis",
    "Title_analysis",
    ]

# ==== DROP COLUMNS ====
df_all_noNA_clean = df_all_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_all_noNA_clean shape: {df_all_noNA_clean.shape}")
print(f"df_all_noNA_clean columns: {df_all_noNA_clean.columns.tolist()}\n")
df_filtered_noNA_clean = df_filtered_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_filtered_noNA_clean shape: {df_filtered_noNA_clean.shape}")
print(f"df_filtered_noNA_clean columns: {df_filtered_noNA_clean.columns.tolist()}\n")

# ==== LOCK DATAFRAMES FOR ANALYSIS ====
print("Final DataFrames for Analysis:")
df_all_final = df_all_noNA_clean.copy()
print(f"df_all_final shape: {df_all_final.shape}")
df_filtered_final = df_filtered_noNA_clean.copy()
print(f"df_filtered_final shape: {df_filtered_final.shape}")

## Exclude rows where Publication Year is 2026?

In [ ]:
choice = input("Excluded rows with Publication Year == 2026? (Y/n): ").strip().lower()

if choice == "n":
    print("Keeping rows with Publication Year == 2026.")
else:
    # drop rows that has "Publication Year" == 2026
    print("df_all_final")
    df_all_final = df_all_final[df_all_final["Publication Year"] != 2026]
    excluded_rows_all = pd.to_numeric(df_all_noNA_clean["Publication Year"], errors="coerce").eq(2026).sum()
    print(f"Number of rows dropped with Publication Year == 2026: {excluded_rows_all}")
    print(f"df_all_final shape after dropping rows with Publication Year == 2026: {df_all_final.shape}")
    print(f"df_all_final has columns: {df_all_final.columns.tolist()}\n")

    print("df_filtered_final")
    df_filtered_final = df_filtered_final[df_filtered_final["Publication Year"] != 2026]
    excluded_rows_filtered = pd.to_numeric(df_filtered_noNA_clean["Publication Year"], errors="coerce").eq(2026).sum()
    print(f"Number of rows dropped with Publication Year == 2026: {excluded_rows_filtered}")
    print(f"df_filtered_final shape after dropping rows with Publication Year == 2026: {df_filtered_final.shape}")
    print(f"df_filtered_final has columns: {df_filtered_final.columns.tolist()}\n")


The DataFrames to use are:
- `df_all_final`
- `df_filtered_final`

## Columns in DataFrames

### `df_all_final`

* abstract_ts
* Timestamp
* Author
* ID
* Publication Year
* keywords_ts
* keywords_facet
* keywords_normalized
* member_id_ss
* primary_member_id_s
* Title
* MASTER THESIS TITLE
* BY
* SUPERVISED BY
* pdf_file
* num_tot_pages
* num_cont_pages
* num_words_full
* num_words_cont
* handin_month
* num_figures
* num_tables
* num_references
* equation_count
* pdf_sha256
* total_sentences
* total_words
* unique_words
* avg_sentence_length
* avg_word_length
* lexical_diversity
* Department_new


### `df_filtered_final`

**COLUMNS RELEVANT FOR ANALYSIS:**
- **`Publication Year`**: The year of publication
- **`MASTER THESIS TITLE`**: The english title of the thesis
- **`BY`**: The author(s) in the format "lastname, name" (if multiple auhtors, they're separated wiht ";") 
- **`SUPERVISED BY`**: The supervisor(s) in the format "lastname, name" (if multiple supervisors, they're separated with ";")
- **`num_tot_pages`**: Number of total pages in .pdf file
- **`num_cont_pages`**: Number of content pages in the .pdf file (excluding appendix, references etc.)
- **`handin_month`**: The month of handin exstracted from the .pdf file. *OBS(!):* disregard the year in the stirng, and use the metric `Publication Year` for true year.
- **`num_figures`**: Number of figures in the .pdf file
- **`num_tables`**: Number of tables in the .pdf file
- **`num_references`** Number of references listed in the section regarding bibliography in the .pdf file
- **`equation_count`**: Number of equations in the .pdf file
- **`total_sentences`**: Number of sentences in main content of .pdf file
- **`total_words`**: Number of words in main content of .pdf file
- **`unique_words`**: Number of unique words in main content of .pdf file
- **`avg_sentence_length`**: Average sentence lenght of main content of .pdf file
- **`avg_word_length`**: Average word lenght of main conent of .pdf file
- **`lexical_diversity`**: Measure of the lexical diversity in the main content of .pdf file (unique_words/total_words)
- **`Department_new`**: The department of DTU from which the thesis is published
- **`grading_scientific_contribution`**: Sub grading score, (x-y)
- **`grading_methodological_rigor`**: Sub grading score, (x-y)
- **`grading_technical_implementation`**: Sub grading score, (x-y)
- **`grading_literature_review`**: Sub grading score, (x-y)
- **`grading_process_professionalism`**: Sub grading score, (x-y)
- **`grading_impact_applicability`**: Sub grading score, (x-y)
- **`grading_research_question_alignment`**: Sub grading score, (x-y)
- **`grading_total_score`**: Total assigned grading score (1-100) for the thesis by local LLM. Consistes of the scores; scientific contribution, methodological rigor, technical implementation, literature review, process professionalism, impact applicability.


**COLUMNS IN DataFrame, BUT NOT RELEVANT FOR ANALYSIS (or is a dublication)**
- Timestamp
- Author
- ID
- member_id_ss
- primary_member_id_s
- Title
- pdf_file
- num_words_full
- num_words_cont
- pdf_sha256
- grading_meta_attempts
- grading_meta_original_chars
- grading_meta_trimmed_at_references
- grading_meta_input_chars
- grading_meta_estimated_input_tokens
- grading_meta_was_truncated
- grading_meta_prompt_fit_attempts
- grading_meta_timeout_seconds
- grading_meta_stream_mode


# General stats

In [ ]:
# print table of df department_new count of each unique
dept_counts = df_filtered_final['Department_new'].value_counts()
#print("--- Thesis Count by Department ---")
#print(dept_counts)

# plot df department_new count of each unique + show count on bars
plt.figure(figsize=(12, 6))
ax = sns.countplot(
    data=df_filtered_final,
    x='Department_new',
    order=df_filtered_final['Department_new'].value_counts().index
)

# add count labels at the end (top) of each bar
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=0, fontsize=9)

plt.xticks(rotation=90)
plt.title('Distribution of Theses by Department')
plt.xlabel('Department')
plt.ylabel('Count of Theses')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
#plt.show()
plt.savefig('exported_plots/2023_trends_analysis/department_distribution.png')

# DEVELOPMENT

# ARCHIVES

## [new legacy element here]

## [new legacy element here]